# Default Risk — Logistic Regression Baseline

Goal: quantify and _rank_ the drivers of default with an interpretable model.
Not chasing max accuracy — logistic regression is chosen because its
coefficients translate directly into "which factors raise/lower default risk
and by how much," which is what a lending team can act on.

Target: default (Charged Off/Default = 1, Fully Paid = 0), ~20% positive rate.


In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/loan.csv"

cols = [
    "loan_amnt", "term", "int_rate", "installment", "grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "purpose", "dti", "delinq_2yrs", "inq_last_6mths", "open_acc",
    "pub_rec", "revol_util", "total_acc", "loan_status",
]

df = pd.read_csv(DATA_PATH, usecols=cols, low_memory=False)
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["default"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)
df = df.drop(columns=["loan_status"])

print("Shape:", df.shape)
print("Default rate:", round(df["default"].mean(), 4))

Shape: (1303638, 18)
Default rate: 0.2007


In [2]:
from sklearn.model_selection import train_test_split

model_df = df.copy()

# Clean text-numeric fields
model_df["term"] = model_df["term"].str.strip().str.replace(" months", "").astype(int)
model_df["revol_util"] = pd.to_numeric(
    model_df["revol_util"].astype(str).str.replace("%", "", regex=False),
    errors="coerce"
)

# emp_length -> numeric years (10+ = 10, <1 = 0)
emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
    "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
    "10+ years": 10,
}
model_df["emp_length"] = model_df["emp_length"].map(emp_map)

# Fill remaining numeric nulls with median (small counts, documented)
num_cols = ["revol_util", "emp_length", "dti", "annual_inc", "delinq_2yrs",
            "inq_last_6mths", "open_acc", "pub_rec", "total_acc"]
for c in num_cols:
    model_df[c] = model_df[c].fillna(model_df[c].median())

# One-hot encode the true categoricals
cat_cols = ["grade", "home_ownership", "verification_status", "purpose"]
model_df = pd.get_dummies(model_df, columns=cat_cols, drop_first=True)

print("Shape after encoding:", model_df.shape)
print("Any nulls left:", model_df.isnull().sum().sum())

Shape after encoding: (1303638, 40)
Any nulls left: 0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X = model_df.drop(columns=["default"])
y = model_df["default"]

# Stratified split preserves the 20% default rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale features so coefficients are comparable
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit. class_weight balances the 80/20 imbalance.
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_s, y_train)

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Model fitted.")

Train rows: 977728
Test rows: 325910
Model fitted.


In [4]:
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Paid", "Default"]))

print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.706

Classification report:
              precision    recall  f1-score   support

        Paid       0.88      0.63      0.74    260488
     Default       0.31      0.67      0.43     65422

    accuracy                           0.64    325910
   macro avg       0.60      0.65      0.58    325910
weighted avg       0.77      0.64      0.67    325910

Confusion matrix (rows=actual, cols=predicted):
[[164078  96410]
 [ 21642  43780]]


In [5]:
# Coefficients = direction + magnitude of each driver (features are scaled, so comparable)
coefs = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

print("Top 10 factors INCREASING default risk:")
print(coefs.head(10).to_string(index=False))

print("\nTop 10 factors DECREASING default risk:")
print(coefs.tail(10).to_string(index=False))

Top 10 factors INCREASING default risk:
    feature  coefficient
    grade_C     0.399110
    grade_D     0.366787
       term     0.289615
    grade_E     0.281460
installment     0.274895
    grade_B     0.252804
    grade_F     0.170008
        dti     0.162938
   int_rate     0.162287
   open_acc     0.125174

Top 10 factors DECREASING default risk:
                feature  coefficient
          purpose_house     0.009066
    purpose_educational     0.007646
   home_ownership_OTHER     0.002517
    home_ownership_NONE    -0.002728
             emp_length    -0.016442
        purpose_wedding    -0.016638
home_ownership_MORTGAGE    -0.092049
              total_acc    -0.124190
             annual_inc    -0.129681
              loan_amnt    -0.220292


In [6]:
# Save ranked coefficients for the Power BI dashboard
import os
os.makedirs("../data/processed", exist_ok=True)
coefs.to_csv("../data/processed/model_coefficients.csv", index=False)
print("Saved:", len(coefs), "feature coefficients")
print("Note: data/processed is gitignored (covered by data/ rule)")

Saved: 39 feature coefficients
Note: data/processed is gitignored (covered by data/ rule)
